# Exercise 1 - Inference Queries

In this exercise, we will answer inference queries from a joint probability distribution.

In the event of a persistent problem, do not hesitate to contact me at

- tobias.pretschold@uni-jena.de

### Submission

- Deadline of submission: Tue, 27 October 11:59 am.
- Submission via Moodle

### Help
In case you cannot solve a task, you can use the saved values within the `help` directory. The checkpoints are named after the corresponding functions; the probability tensor from Task 2 is saved as `prob_tensor.npy`.

- Load arrays with [Numpy](https://numpy.org/doc/stable/reference/generated/numpy.load.html)
```
np.load('help/array_name.npy')
```
- Load functions with [Dill](https://dill.readthedocs.io/en/latest/dill.html)
```
import dill
with open('help/some_func.pkl', 'rb') as f:
    func = dill.load(f)
```

to continue working on the other tasks.

In [18]:
import numpy as np
import matplotlib.pyplot as plt

# Probability Table

We will use a probability table derived from the [migraine dataset](https://www.kaggle.com/datasets/weinoose/migraine-classification).
Due to the lack of description, we can only guess what some of the attributes stand for exactly.
Nevertheless, we have 12 discrete features from patients suffering from migraine.

1. `Age`: the age of the patient in intervals of 20 years
    - 0: $< 20$
    - 1: $\in [20, 40)$
    - 2: $\in [40, 60)$
    - 3: $\geq 60$
2. `Duration`: how long did the migraine attack last?
    - 0: short
    - 1: normal
    - 2: long
3. `Intensity`: how intense was the migraine attack?
    - 0: very light
    - 1: light
    - 2: intense
    - 3: very intense
4. `Nausea`: did the patient feel sick?
    - 0: no
    - 1: yes
5. `Vomit`: did the migraine attack cause the patient to vomit?
    - 0: no
    - 1: yes
6. `Phonophobia`: did the patient feel specifically sensitive to sound?
    - 0: no
    - 1: yes
7. `Photophobia`: did the patient feel specifically sensitive to light?
    - 0: no
    - 1: yes
8. `Vertigo`: did the patient experience vertigo?
    - 0: no
    - 1: yes
9. `Tinnitus`: did the patient suffer from tinnitus?
    - 0: no
    - 1: yes
10. `Conscience`: did the patient lose consciousness?
    - 0: no
    - 1: yes
11. `Paresthesia`: did the patient feel numbness?
    - 0: no
    - 1: yes
12. `Type`: What kind of migraine did the patient have?
    - 0: Basilar-type aura
    - 1: Familial hemiplegic migraine
    - 2: Migraine without aura
    - 3: Other
    - 4: Sporadic hemiplegic migraine
    - 5: Typical aura with migraine
    - 6: Typical aura without migraine

### Task 1

Load the probability table from `prob_table.npy`.
The first 12 columns correspond to the features, the last column holds the probability.

In [19]:
import numpy as np
import matplotlib.pyplot as plt

columns = ['Age',
           'Duration',
           'Intensity',
           'Nausea',
           'Vomit',
           'Phonophobia',
           'Photophobia',
           'Vertigo',
           'Tinnitus',
           'Conscience',
           'Paresthesia',
           'Type']
probability_table = np.load("prob_table.npy")
print(probability_table.shape) # rows and columns of the probability table
print(probability_table[5:10]) # first 5 rows of the probability table

(86016, 13)
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 5.00000000e+00
  1.15987752e-05]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 6.00000000e+00
  1.15987752e-05]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00
  1.15987752e-05]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.00000000e+00 1.00000000e+00
  1.15987752e-05]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.00000000e+00 2

# Probability Tensor

The flattened table is not the most natural representation of a joint distribution.

From now on we treat the 12 features as random variables $X_1, \dots, X_{12}$ as in the probability basics sheet. There, $X_i$ also denotes the space of values of the $i$-th variable, which here is the set of integer codes listed above, e.g. $X_2 = \{0, 1, 2\}$ for `Duration`. The joint distribution $p$ of $X_1, \dots, X_{12}$ is then a probability distribution over the product space $\mathcal{X} = X_1 \times \dots \times X_{12}$.

Because all values are integers starting at 0, a joint value $x = (x_1, \dots, x_{12}) \in \mathcal{X}$ can directly be used as an index into a tensor. The natural representation of $p$ is therefore a 12-dimensional tensor $P$ with one axis per variable, such that

\begin{equation}
P[x_1, \dots, x_{12}] = p(x_1, \dots, x_{12})\,.
\end{equation}

### Task 2

Convert the flattened table into a probability tensor `P`.

As a sanity check, `P` should have the shape `(4, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7)` and sum to 1.

In [20]:
shape = []

for i in range(12):
    column = probability_table[:, i] # load the i-th column of the probability table
    max_value = column.max() # find the maximum value in the column
    dimension_size = int(max_value) + 1 # add 1 to the maximum value to get the size of the dimension
    shape.append(dimension_size) # append the size of the dimension to the shape list

shape = tuple(shape)
print(shape)

p = np.zeros(shape) # create a zero array with the shape of the probability table (tensor)

for row in probability_table:
    indices = tuple(row[:12].astype(int)) # convert the first 12 columns of the row to a tuple of integers
    #print(indices) # print the indices for debugging
    p[indices] = row[12] # assign the probability value to the corresponding indices
    #print(p[indices]) # print the probability value for debugging

print(p.sum()) # check if the sum of the probabilities is 1



(4, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7)
1.0


# Inference Queries

The probability tensor encodes the joint probability distribution of the variables $X_1, \dots, X_{12}$. The whole point of having such a distribution is to answer queries with it. As in the probability basics sheet, we write $A, B \subseteq \{1, \dots, 12\}$ for subsets of variable indices, $X_A = \prod_{i \in A} X_i$ for the associated space with values $x_A \in X_A$, and $\bar{A} = \{1, \dots, 12\} \setminus A$ for the complement of $A$.

Throughout this exercise, we use the convention that the axes of a marginal tensor are ordered by ascending variable index. For simplicity, you may assume that all index sets are given in ascending order and that $A$ and $B$ are disjoint. Note that the math below indexes the variables from 1 to 12, while in code the corresponding tensor axes are 0 to 11.

## Marginal Distribution
For the marginal, a subset of indices $A$ is given and the marginal distribution

\begin{equation}
p(x_A) = \sum_{x_{\bar{A}} \in X_{\bar{A}}} p(x_A, x_{\bar{A}})
\end{equation}

has to be computed, that is, one probability for each value $x_A \in X_A$, i.e. a tensor with one axis per variable in $A$.

### Task 3

Calculate the marginal distribution of `Vomit` and `Vertigo`.

As a sanity check, the probability for `Vomit`=0 and `Vertigo`=0 should be `0.25076552`

In [21]:

def marginal(P: np.ndarray, A: list) -> np.ndarray:
    '''
    Computes the marginal distribution p(x_A) of the variables in A.

    @Params:
        P... probability tensor with one axis per variable
        A... list of variable indices

    @Returns:
        probability tensor with one axis per variable in A
    '''
    # Create a list of axes to sum over
    axes_to_sum = []
    for i in range(P.ndim):
        if i not in A:
            axes_to_sum.append(i)
    return P.sum(axis=tuple(axes_to_sum))

print(type(p))
print(p.ndim)
result = marginal(p,[4,7])
print(result)


<class 'numpy.ndarray'>
12
[[0.25076552 0.24964624]
 [0.25010439 0.24948385]]


## Conditional Distribution
For the conditional, two disjoint subsets of indices $A, B$ together with an observed value $x_B \in X_B$ are given and the conditional distribution

\begin{equation}
p(x_A \mid x_B) = \frac{p(x_A, x_B)}{p(x_B)}
\end{equation}

has to be computed, which is again a tensor with one axis per variable in $A$, while the observed $x_B$ stays fixed.

### Task 4
Calculate the conditional distribution of `Type` given we observe the patient feels sick (`Nausea`=1) but has no tinnitus (`Tinnitus`=0).

As a sanity check, the probability for `Type` = 0 should be `0.14188566`

In [ ]:
def conditional(P: np.ndarray, A: list, B: list, x_B: list) -> np.ndarray:
    '''
    Computes the conditional distribution p(x_A | x_B) of the variables in A given the observed values x_B.

    @Params:
        P... probability tensor with one axis per variable
        A... list of variable indices (query)
        B... list of variable indices (evidence)
        x_B... list of observed values for the variables in B

    @Returns:
        probability tensor with one axis per variable in A
    '''
    kept_axes =  sorted(A + B) # sort the list of axes to keep
    joint = marginal(P, kept_axes) # compute the joint distribution p(x_A, x_B)

    print ("joint: start", joint, "endjoint") # print the shape of the joint distribution for debugging

    indices = []
    for i in kept_axes:
        if i in B:
            position = B.index(i) # find the position of the variable in B
            indices.append(x_B[position]) # append the observed value to the indices list
        else:
            indices.append(slice(None)) # append a slice(None) to the indices list for the variables in A
    probablitites = joint[tuple(indices)] # index the joint distribution with the indices list to get p(x_A, x_B=x_B)
    print ("probabilities: ", probablitites) # print the shape of the probabilities for debugging

    return probablitites / probablitites.sum() # normalize the probabilities to get p(x_A | x_B=x_B)

result = conditional(p, [11], [3,8], [1,0]) 
#print(result) # print the conditional distribution for Type given Nausea=1 and Tinnitus=0  
print(result[0]) # print the probability of Type=0 given Nausea=1 and Tinnitus=0


joint:  [[[0.03563144 0.03563144 0.03563144 0.03564884 0.03563144 0.03563144
   0.03563724]
  [0.03563144 0.03563144 0.03563144 0.03563724 0.03563144 0.03563144
   0.03563144]]

 [[0.03569523 0.03573003 0.0359794  0.03569523 0.03568943 0.03706389
   0.03572423]
  [0.03567203 0.03567203 0.03563144 0.03564304 0.03565463 0.03563144
   0.03564884]]] endjoint
probabilities:  [0.03569523 0.03573003 0.0359794  0.03569523 0.03568943 0.03706389
 0.03572423]
0.14188566159520538


## Probability of Evidence

Often we are not interested in a whole distribution, but only in the probability of one concrete observation. For disjoint subsets of indices $A, B$ and values $x_A \in X_A$, $x_B \in X_B$, the single number

\begin{equation}
p(x_A \mid x_B)
\end{equation}

has to be computed. In the special case $B = \emptyset$, compute the marginal probability $p(x_A)$ instead.

### Task 5

Calculate the probability of a short attack (`Duration` = 0) given the patient is younger than 20 years (`Age` = 0) and experiences dizziness (`Vertigo` = 1).

As a sanity check, the wanted probability should be around `0.333`

In [6]:
def prob_of_evidence(P: np.ndarray, A: list, x_A: list, B: list, x_B: list) -> float:
    '''
    Computes the probability p(x_A | x_B) of observing the values x_A given the observed values x_B.

    @Params:
        P... probability tensor with one axis per variable
        A... list of variable indices (query)
        x_A... list of values for the variables in A
        B... list of variable indices (evidence)
        x_B... list of observed values for the variables in B

    @Returns:
        probability of x_A given x_B
    '''
    # With evidence, use the conditional distribution; otherwise use the marginal.
    if len(B) > 0:
        distribution = conditional(P, A, B, x_B)
    else:
        distribution = marginal(P, A)

    # Select the requested values of the query variables.
    return float(distribution[tuple(x_A)])


# Duration is axis 1, Age is axis 0, and Vertigo is axis 7.
probability = prob_of_evidence(p, [1], [0], [0, 7], [0, 1])
print("Probability of a short attack given Age = 0 and Vertigo = 1:", probability)


joint: start [[[0.04172659 0.04158161]
  [0.0416686  0.04157581]
  [0.04161061 0.04158161]]

 [[0.04224274 0.0416918 ]
  [0.04192377 0.04159321]
  [0.04174979 0.04161641]]

 [[0.04188898 0.04158741]
  [0.0416744  0.04157001]
  [0.0416396  0.0416222 ]]

 [[0.04158741 0.04157001]
  [0.04157581 0.04157001]
  [0.04158161 0.04157001]]] endjoint
probabilities:  [0.04158161 0.04157581 0.04158161]
Probability of a short attack given Age = 0 and Vertigo = 1: 0.33334883072202337


## Most Probable Explanation (MPE)

Suppose we observe the value $x_B \in X_B$ for a subset of indices $B$. In this case, $x_B$ is sometimes called evidence. The most probable explanation of the evidence is the most probable joint value of all remaining variables,

\begin{equation}
\hat{x}_{\bar{B}} = \text{argmax}_{x_{\bar{B}} \in X_{\bar{B}}}\, p(x_{\bar{B}} \mid x_B)\,.
\end{equation}

Together with the evidence $x_B$ itself, it forms a full configuration $\hat{x} \in \mathcal{X}$. Since the denominator of $p(x_{\bar{B}} \mid x_B) = p(x_{\bar{B}}, x_B) \, / \, p(x_B)$ does not depend on $x_{\bar{B}}$, the MPE equivalently maximizes the joint probability $p(x_{\bar{B}}, x_B)$.

### Task 6

What is the intensity of the most probable explanation for a long (`Duration` = 2) migraine attack of a 30-year-old (`Age` = 1) patient with tinnitus (`Tinnitus` = 1), both Phono- and Photophobia (`Phonophobia` = 1, `Photophobia` = 1) where we know that it is of the type "Basilar-type aura" (`Type` = 0)?

As a sanity check, the intensity of the MPE should be `3`.

In [7]:
def most_prob_explanation(P: np.ndarray, B: list, x_B: list) -> np.ndarray:
    '''
    Computes the most probable explanation of the evidence x_B.

    @Params:
        P... probability tensor with one axis per variable
        B... list of variable indices (evidence)
        x_B... list of observed values for the variables in B

    @Returns:
        most probable full configuration x with x[B] = x_B
    '''
    # Fix the evidence and keep all values for the remaining variables.
    indices = []
    remaining_axes = []

    for axis in range(P.ndim):
        if axis in B:
            position = B.index(axis)
            indices.append(x_B[position])
        else:
            indices.append(slice(None))
            remaining_axes.append(axis)

    probabilities = P[tuple(indices)]

    # Find the largest entry and convert its flat index to tensor coordinates.
    flat_index = np.argmax(probabilities)
    best_values = np.unravel_index(flat_index, probabilities.shape)

    # Rebuild the full configuration, including the observed values.
    explanation = np.zeros(P.ndim, dtype=int)

    for i in range(len(B)):
        explanation[B[i]] = x_B[i]

    for i in range(len(remaining_axes)):
        explanation[remaining_axes[i]] = best_values[i]

    return explanation


# Evidence: Age, Duration, Phonophobia, Photophobia, Tinnitus, Type.
evidence_axes = [0, 1, 5, 6, 8, 11]
evidence_values = [1, 2, 1, 1, 1, 0]
mpe = most_prob_explanation(p, evidence_axes, evidence_values)
print("Most probable full configuration:", mpe)
print("Intensity of the MPE:", mpe[2])


Most probable full configuration: [1 2 3 1 0 1 1 0 1 0 0 0]
Intensity of the MPE: 3


## Maximum a Posteriori Hypothesis (MAP)

Often only a few variables are of interest. For disjoint subsets of indices $A, B$ and an observed value $x_B \in X_B$, the maximum a posteriori hypothesis for the variables in $A$ is

\begin{equation}
\hat{x}_A = \text{argmax}_{x_A \in X_A}\, p(x_A \mid x_B)\,.
\end{equation}

The name comes from Bayes' rule: $p(x_A \mid x_B)$ is the *posterior* distribution of $x_A$ after observing the evidence $x_B$. In contrast to the MPE, all variables outside $A \cup B$ are marginalized out before maximizing. For $A = \bar{B}$ the MAP hypothesis coincides with the MPE, but in general it is not simply the restriction of the MPE to the variables in $A$, since marginalization and maximization do not commute.

### Task 7
What is the maximum a posteriori hypothesis for the type of migraine attack (`Type`) of a 15-year-old person (`Age` = 0) that has tinnitus (`Tinnitus` = 1)?

As a sanity check, the MAP hypothesis should be `1`.

In [8]:
def max_a_posteriori(P: np.ndarray, A: list, B: list, x_B: list) -> np.ndarray:
    '''
    Computes the maximum a posteriori hypothesis for the variables in A given the evidence x_B.

    @Params:
        P... probability tensor with one axis per variable
        A... list of variable indices (query)
        B... list of variable indices (evidence)
        x_B... list of observed values for the variables in B

    @Returns:
        x_A that maximizes p(x_A | x_B)
    '''
    # Sum out the other variables and condition on the evidence.
    probabilities = conditional(P, A, B, x_B)

    # Find the most likely combination of values for the query variables.
    flat_index = np.argmax(probabilities)
    best_values = np.unravel_index(flat_index, probabilities.shape)

    return np.array(best_values, dtype=int)


# Query Type (axis 11), given Age = 0 and Tinnitus = 1.
map_hypothesis = max_a_posteriori(p, [11], [0, 8], [0, 1])
print("Most likely migraine type:", map_hypothesis[0])


joint: start [[[0.01781572 0.01785051 0.01787371 0.01781572 0.01782732 0.0180071
   0.01782152]
  [0.01781572 0.01782732 0.01781572 0.01781572 0.01782152 0.01781572
   0.01782152]]

 [[0.01787371 0.01787951 0.0180129  0.01785051 0.01786211 0.01862183
   0.01790851]
  [0.01785051 0.01784472 0.01781572 0.01782732 0.01783312 0.01781572
   0.01782152]]

 [[0.01782152 0.01781572 0.01789111 0.01786211 0.01781572 0.01823327
   0.01781572]
  [0.01782152 0.01781572 0.01781572 0.01782152 0.01781572 0.01781572
   0.01782152]]

 [[0.01781572 0.01781572 0.01783312 0.01781572 0.01781572 0.01783312
   0.01781572]
  [0.01781572 0.01781572 0.01781572 0.01781572 0.01781572 0.01781572
   0.01781572]]] endjoint
probabilities:  [0.01781572 0.01782732 0.01781572 0.01781572 0.01782152 0.01781572
 0.01782152]
Most likely migraine type: 1
